# FedKG-Maint — Phase 3 Privacy Closure Experiment

This add-on closes the privacy/utility question without re-running every Phase 3 baseline.

**Protocol:** (1) run a DP-FedAvg noise sweep for seeds 42/43/44 on `validation` only, (2) choose the best mean validation PR-AUC among candidates meeting the stated client-update privacy bound `epsilon <= 10` at `delta=1e-5`, (3) lock the noise multiplier and per-seed checkpoints, and (4) evaluate that selected DP configuration on `test_site_full` once. It also compares the selected DP results with the previously recorded FedAvg, CentralMLP, and default-noise DP-FedAvg outputs.

The epsilon limit of 10 is an **experimental acceptance cap for this study**, not a universal privacy standard. The accountant is scoped to the clipped/noised client-update channel under a trusted server; it does not provide end-to-end DP for validation-based checkpoint selection, normalization statistics, raw-data access, or diagnostic releases. `SecureAggSim-FedAvg` from the main notebook remains an arithmetic demonstration, not cryptographic secure aggregation.

Run Phase 1 v3, Phase 2 v3, and Phase 3 v2 first. This add-on writes to a new directory and does not overwrite their files. It uses only the `full` feature set with `global` normalization for the privacy selection/evaluation protocol; other configurations remain available in the Phase 3 main report.

## 1. Setup and artifact checks

The model, optimizer, batching and clipping settings below match Phase 3 v2. The DP sweep is restricted to train/validation. The held-out test is not loaded until Section 4, after the noise choice is locked.

In [ ]:
import os, sys, json, math, time, random, warnings, importlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.utils import parameters_to_vector, vector_to_parameters
from sklearn.metrics import average_precision_score

warnings.filterwarnings("ignore", category=FutureWarning)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/FedKG-Maint")
except ImportError:
    PROJECT_DIR = Path(os.environ.get("FEDKG_ROOT", "./FedKG-Maint")).resolve()

PREP_DIR = PROJECT_DIR / "prep_v3"
P2_DIR = PROJECT_DIR / "phase2_baselines_prep_v3"
P3_DIR = PROJECT_DIR / "phase3_federated_prep_v3_privacy_v2"
OUT_DIR = PROJECT_DIR / "phase3_federated_privacy_closure_v1"
for sub in ["models", "predictions", "results", "reports", "logs"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

required = [
    PREP_DIR / "fedkg_data.py",
    PREP_DIR / "configs" / "feature_groups.json",
    P2_DIR / "fedkg_eval.py",
    P3_DIR / "results" / "federated_results.csv",
    P3_DIR / "reports" / "phase3_federated_report.json",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing prerequisites; run Phase 1 v3, Phase 2 v3, and Phase 3 v2 first:\n" + "\n".join(missing))

sys.path.insert(0, str(PREP_DIR)); sys.path.insert(0, str(P2_DIR))
sys.modules.pop("fedkg_data", None); sys.modules.pop("fedkg_eval", None)
importlib.invalidate_caches()
import fedkg_data
from fedkg_eval import SetEval, draw_weights, ci
FEATURE_CFG = json.loads((PREP_DIR / "configs" / "feature_groups.json").read_text())
TARGET = FEATURE_CFG["target"]
SEEDS = [42, 43, 44]
CLIENTS = ["factory_1", "factory_2", "factory_3", "factory_4"]
HELD_OUT = "factory_5"
MAX_ROUNDS = 12
BATCH_SIZE = 4096
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
POS_WEIGHT = 20.0
HIDDEN = [64, 32]
DROPOUT = 0.10
CLIP_NORM = 1.0
DELTA = 1e-5
EPSILON_CAP = 10.0  # study-specific acceptance cap, not a universal standard
NOISE_CANDIDATES = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 4.0]
BOOTSTRAP_B = 1000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
P2_REPORT = json.loads((P2_DIR / "reports" / "phase2_baselines_report.json").read_text())
P3_BASE = pd.read_csv(P3_DIR / "results" / "federated_results.csv")
print("Device:", DEVICE)
print("Project:", PROJECT_DIR)
print("Output:", OUT_DIR)
print("Seeds:", SEEDS, "| rounds:", MAX_ROUNDS, "| clients:", CLIENTS)
print("Noise candidates:", NOISE_CANDIDATES, "| epsilon cap:", EPSILON_CAP, "| delta:", DELTA)


## 2. Matching training utilities

The accountant uses full client participation, no sampling amplification, RDP composition over all rounds, and the same discrete RDP orders as Phase 3 v2. The client update is clipped in parameter space before adding Gaussian noise to the aggregate.

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def make_model(n_features, seed):
    torch.manual_seed(int(seed))
    h1, h2 = HIDDEN
    return nn.Sequential(
        nn.Linear(n_features, h1), nn.ReLU(), nn.Dropout(DROPOUT),
        nn.Linear(h1, h2), nn.ReLU(), nn.Dropout(DROPOUT),
        nn.Linear(h2, 1)
    ).to(DEVICE)

def cpu_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def update_vector(model):
    return parameters_to_vector([p.detach() for p in model.parameters()]).detach().clone()

def load_vector(model, vector):
    vector_to_parameters(vector, model.parameters())

def data_tensor(X, y=None):
    xt = torch.as_tensor(np.asarray(X, dtype=np.float32), dtype=torch.float32, device=DEVICE)
    if y is None: return xt
    yt = torch.as_tensor(np.asarray(y, dtype=np.float32), dtype=torch.float32, device=DEVICE).reshape(-1, 1)
    return xt, yt

def train_one_epoch(model, X_t, y_t, row_indices, optimizer):
    model.train()
    order = row_indices[torch.randperm(len(row_indices), device=DEVICE)]
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    total_loss, seen = 0.0, 0
    for start in range(0, len(order), BATCH_SIZE):
        idx = order[start:start+BATCH_SIZE]
        xb = X_t.index_select(0, idx); yb = y_t.index_select(0, idx)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(xb), yb)
        loss.backward(); optimizer.step()
        n = len(idx); total_loss += float(loss.detach().cpu()) * n; seen += n
    return total_loss / max(seen, 1)

@torch.no_grad()
def predict_scores(model, X, batch_size=32768):
    model.eval()
    xt = torch.as_tensor(np.asarray(X, dtype=np.float32), dtype=torch.float32, device=DEVICE)
    out=[]
    for start in range(0, len(xt), batch_size):
        out.append(torch.sigmoid(model(xt[start:start+batch_size]).reshape(-1)).detach().cpu().numpy())
    return np.concatenate(out) if out else np.empty(0, dtype=np.float64)

def privacy_epsilon(noise_multiplier, steps=MAX_ROUNDS, delta=DELTA):
    if noise_multiplier <= 0: return float("inf"), None
    orders = np.array([1.25, 1.5, 2, 3, 4, 5, 8, 10, 16, 32, 64, 128, 256, 512], dtype=float)
    rdp = steps * orders / (2.0 * noise_multiplier**2)
    eps = rdp + math.log(1.0/delta)/(orders-1.0)
    i = int(np.argmin(eps))
    return float(eps[i]), float(orders[i])

def train_dp_candidate(Xtr, ytr, mtr, Xv, yv, seed, noise_multiplier):
    seed_everything(seed)
    n_features = Xtr.shape[1]
    X_t, y_t = data_tensor(Xtr, ytr)
    client_idx = {
        c: torch.as_tensor(np.flatnonzero(mtr["client_id"].to_numpy() == c), dtype=torch.long, device=DEVICE)
        for c in CLIENTS
    }
    sizes = {c: len(client_idx[c]) for c in CLIENTS}
    if any(v == 0 for v in sizes.values()) or len(set(sizes.values())) != 1:
        raise ValueError(f"Expected four equal, non-empty clients; got {sizes}")
    model = make_model(n_features, seed)
    best = {"ap": -np.inf, "round": None, "state": None, "scores": None, "threshold": None}
    history=[]; t0=time.time()
    for rnd in range(1, MAX_ROUNDS+1):
        base = update_vector(model)
        deltas=[]
        for client in CLIENTS:
            local = make_model(n_features, seed + rnd*100 + int(client.split("_")[-1]))
            local.load_state_dict(model.state_dict())
            opt = torch.optim.Adam(local.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
            train_one_epoch(local, X_t, y_t, client_idx[client], opt)
            deltas.append(update_vector(local) - base)
            del local, opt
        clipped=[]
        for d in deltas:
            norm = torch.linalg.vector_norm(d)
            factor = torch.clamp(torch.tensor(CLIP_NORM, device=DEVICE)/(norm+1e-12), max=1.0)
            clipped.append(d*factor)
        mean_delta = torch.stack(clipped, dim=0).mean(dim=0)
        noise_std = float(noise_multiplier) * CLIP_NORM / len(CLIENTS)
        mean_delta = mean_delta + torch.normal(0.0, noise_std, size=mean_delta.shape, device=DEVICE)
        load_vector(model, base + mean_delta)
        val_scores = predict_scores(model, Xv)
        val_ap = float(average_precision_score(yv, val_scores))
        packed = VAL_EVAL.pack(val_scores)
        threshold = float(VAL_EVAL.best_f1_threshold(packed))
        history.append({"seed": seed, "noise_multiplier": float(noise_multiplier), "round": rnd, "validation_ap": val_ap})
        if val_ap > best["ap"]:
            best = {"ap": val_ap, "round": rnd, "state": cpu_state(model), "scores": val_scores.copy(), "threshold": threshold}
    eps, order = privacy_epsilon(noise_multiplier)
    return {
        "seed": seed, "noise_multiplier": float(noise_multiplier), "epsilon": eps, "rdp_order": order,
        "delta": DELTA, "validation_ap": float(best["ap"]), "selected_round": int(best["round"]),
        "threshold": float(best["threshold"]), "state": best["state"], "history": history,
        "fit_seconds": time.time()-t0, "n_features": n_features,
        "expected_noise_l2_ratio": float(noise_multiplier*np.sqrt(sum(p.numel() for p in model.parameters()))/len(CLIENTS))
    }
print("[PASS] DP closure training utilities are defined.")


## 3. Validation-only sweep, across all three seeds

This cell loads only the train/validation split for the primary `full/global` configuration. Candidate models are selected by validation PR-AUC; the noise multiplier is then chosen using mean validation AP among candidates with `epsilon <= 10`. The epsilon cap was adopted as the closure experiment's acceptance rule; it is not presented as a universal policy threshold.

In [ ]:
Xtr, ytr, mtr = fedkg_data.load_split("train", policy="global", feature_set="full")
Xv, yv, mv = fedkg_data.load_split("validation", policy="global", feature_set="full")
assert set(mtr["client_id"].unique()) == set(CLIENTS)
assert HELD_OUT not in set(mtr["client_id"].unique())
assert HELD_OUT not in set(mv["client_id"].unique())
assert np.isfinite(Xtr).all() and np.isfinite(Xv).all()
# Stream the raw failure log and retain validation-machine events only; held-out events are not materialized during selection.
val_machines = set(mv["machineID"].unique())
_val_chunks = []
for _chunk in pd.read_csv(PROJECT_DIR / "data" / "raw" / "PdM_failures.csv", usecols=["machineID", "datetime"], chunksize=100_000):
    _chunk = _chunk[_chunk["machineID"].isin(val_machines)].copy()
    if len(_chunk): _val_chunks.append(_chunk)
val_failures = pd.concat(_val_chunks, ignore_index=True) if _val_chunks else pd.DataFrame(columns=["machineID", "datetime"])
val_failures["datetime"] = pd.to_datetime(val_failures["datetime"])
val_failures = val_failures.drop_duplicates().sort_values(["machineID", "datetime"])
VAL_EVAL = SetEval(mv.reset_index(drop=True), yv, val_failures)

SWEEP_ARTIFACTS=[]; SWEEP_HISTORY=[]
for seed in SEEDS:
    for sigma in NOISE_CANDIDATES:
        started = time.time()
        art = train_dp_candidate(Xtr, ytr, mtr, Xv, yv, seed, sigma)
        SWEEP_ARTIFACTS.append(art)
        SWEEP_HISTORY.extend(art["history"])
        print(f"seed={seed} sigma={sigma:>4} epsilon={art['epsilon']:.3f} val_AP={art['validation_ap']:.4f} best_round={art['selected_round']} time={time.time()-started:.1f}s")

SWEEP = pd.DataFrame([{k: a[k] for k in ["seed", "noise_multiplier", "epsilon", "rdp_order", "delta", "validation_ap", "selected_round", "fit_seconds", "expected_noise_l2_ratio"]} for a in SWEEP_ARTIFACTS])
SWEEP.to_csv(OUT_DIR / "results" / "dp_noise_sweep_by_seed_validation_only.csv", index=False)
pd.DataFrame(SWEEP_HISTORY).to_csv(OUT_DIR / "logs" / "dp_noise_sweep_learning_curves.csv", index=False)
SWEEP_SUMMARY = (SWEEP.groupby("noise_multiplier")
                 .agg(epsilon=("epsilon", "first"), mean_validation_ap=("validation_ap", "mean"),
                      sd_validation_ap=("validation_ap", "std"), seeds=("seed", "nunique"),
                      mean_selected_round=("selected_round", "mean"),
                      noise_l2_ratio=("expected_noise_l2_ratio", "first"))
                 .reset_index().sort_values("noise_multiplier"))
SWEEP_SUMMARY["eligible_epsilon_cap"] = SWEEP_SUMMARY["epsilon"] <= EPSILON_CAP
SWEEP_SUMMARY.to_csv(OUT_DIR / "results" / "dp_noise_sweep_summary_validation_only.csv", index=False)
print("\nValidation-only noise sweep summary:")
print(SWEEP_SUMMARY.to_string(index=False, float_format=lambda z: f"{z:.4f}"))

eligible = SWEEP_SUMMARY[SWEEP_SUMMARY["eligible_epsilon_cap"]].copy()
if eligible.empty:
    raise RuntimeError("No DP noise candidate satisfies the predefined epsilon cap. Widen candidates or revise the study protocol before opening test data.")
selected_sigma = float(eligible.sort_values(["mean_validation_ap", "noise_multiplier"], ascending=[False, True]).iloc[0]["noise_multiplier"])
SELECTED = {a["seed"]: a for a in SWEEP_ARTIFACTS if abs(a["noise_multiplier"]-selected_sigma) < 1e-12}
assert set(SELECTED) == set(SEEDS)
selection_record = {
    "selection_split": "validation only",
    "feature_set": "full", "normalization": "global",
    "epsilon_cap": EPSILON_CAP, "delta": DELTA,
    "selected_noise_multiplier": selected_sigma,
    "epsilon_bound_training_update_channel": privacy_epsilon(selected_sigma)[0],
    "mean_validation_ap": float(eligible.loc[np.isclose(eligible["noise_multiplier"], selected_sigma), "mean_validation_ap"].iloc[0]),
    "all_candidates_and_seed_results": SWEEP_SUMMARY.to_dict("records"),
    "test_split_loaded_before_selection": False,
    "only_validation_machine_failure_events_retained_during_selection": True,
}
(OUT_DIR / "reports" / "dp_selection_lock.json").write_text(json.dumps(selection_record, indent=2))
for seed, art in SELECTED.items():
    torch.save({"state": art["state"], "method": "DP-FedAvg", "feature_set": "full", "policy": "global",
                "seed": seed, "n_features": art["n_features"], "noise_multiplier": selected_sigma,
                "epsilon": art["epsilon"], "delta": DELTA, "selected_round": art["selected_round"],
                "validation_ap": art["validation_ap"], "threshold": art["threshold"],
                "config": {"max_rounds": MAX_ROUNDS, "clip_norm": CLIP_NORM, "noise_multiplier": selected_sigma,
                           "batch_size": BATCH_SIZE, "pos_weight": POS_WEIGHT}}
               , OUT_DIR / "models" / f"selected_DP_FedAvg_full_global_seed{seed}.pt")
print(f"\n[LOCKED] selected noise multiplier={selected_sigma}; epsilon={privacy_epsilon(selected_sigma)[0]:.3f}; this choice used validation only.")
print("[PASS] No test split was loaded in the sweep/selection stage.")


## 4. One final evaluation of the locked DP configuration

This section loads `test_site_full` only after the noise multiplier and all per-seed checkpoints have been locked. It scores each seed, computes machine-cluster bootstrap confidence intervals, and compares against the existing Phase 3 results for FedAvg, CentralMLP, and default-noise DP-FedAvg. Test results are descriptive only and do not feed back into the selection above.

In [ ]:
# First test-set load in this add-on: selection is already locked above.
Xtest, ytest, mtest = fedkg_data.load_split("test_site_full", policy="global", feature_set="full")
assert set(mtest["client_id"].unique()) == {HELD_OUT}
raw_failures_all = pd.read_csv(PROJECT_DIR / "data" / "raw" / "PdM_failures.csv", parse_dates=["datetime"])
FAIL_EVENTS = (raw_failures_all[["machineID", "datetime"]].drop_duplicates()
               .sort_values(["machineID", "datetime"]).reset_index(drop=True))
TEST_EVAL = SetEval(mtest.reset_index(drop=True), ytest, FAIL_EVENTS)

FINAL_ROWS=[]; FINAL_REPS={}; FINAL_PRED={}
for seed in SEEDS:
    art = SELECTED[seed]
    model = make_model(art["n_features"], seed)
    model.load_state_dict(art["state"])
    scores = predict_scores(model, Xtest)
    packed = TEST_EVAL.pack(scores)
    metrics = TEST_EVAL.metrics(packed, None, art["threshold"])
    W = draw_weights(TEST_EVAL.M, BOOTSTRAP_B, 90131 + seed)
    reps = TEST_EVAL.bootstrap(packed, art["threshold"], W)
    lo, hi = ci(reps["ap"].dropna().to_numpy(), 0.95)
    FINAL_REPS[seed] = reps; FINAL_PRED[seed] = scores.astype(np.float32)
    FINAL_ROWS.append({"seed": seed, "method": "DP-FedAvg-selected", "noise_multiplier": selected_sigma,
                       "epsilon_bound_training_update_channel": art["epsilon"], "delta": DELTA,
                       "selected_round_validation_only": art["selected_round"], "validation_ap": art["validation_ap"],
                       "test_site_full_ap": metrics["ap"], "ap_ci_low": lo, "ap_ci_high": hi,
                       "roc_auc": metrics["roc_auc"], "precision": metrics["precision"], "recall": metrics["recall"],
                       "f1": metrics["f1"], "fpr": metrics["fpr"], "event_recall": metrics["event_recall"],
                       "test_prevalence": float(np.mean(ytest)), "n_machines": TEST_EVAL.M, "n_failure_events": TEST_EVAL.n_events})
    pd.DataFrame({"machineID": mtest["machineID"].to_numpy(), "datetime": mtest["datetime"].to_numpy(),
                  "score": scores.astype(np.float32), "seed": seed}).to_parquet(
        OUT_DIR / "predictions" / f"selected_DP_FedAvg_seed{seed}_test_site_full.parquet", index=False)

FINAL = pd.DataFrame(FINAL_ROWS)
FINAL.to_csv(OUT_DIR / "results" / "selected_dp_final_test_results.csv", index=False)
print(FINAL.to_string(index=False, float_format=lambda z: f"{z:.4f}"))

# Mean-test performance compared descriptively against previously saved primary-set results.
BASE = P3_BASE[(P3_BASE["eval_set"] == "test_site_full") & (P3_BASE["status"] == "scored") &
               (P3_BASE["feature_set"] == "full") & (P3_BASE["policy"] == "global")].copy()
BASE_KEEP = BASE[BASE["method"].isin(["FedAvg", "CentralMLP", "DP-FedAvg"])][["method", "seed", "ap", "ap_ci_low", "ap_ci_high"]]
base_summary = (BASE_KEEP.groupby("method").agg(mean_ap=("ap", "mean"), sd_ap=("ap", "std"), seeds=("seed", "nunique")).reset_index())
selected_summary = pd.DataFrame([{"method": "DP-FedAvg-selected", "mean_ap": FINAL["test_site_full_ap"].mean(),
                                 "sd_ap": FINAL["test_site_full_ap"].std(ddof=1), "seeds": FINAL["seed"].nunique()}])
comparison = pd.concat([base_summary, selected_summary], ignore_index=True)
comparison.to_csv(OUT_DIR / "results" / "selected_dp_vs_phase3_baselines.csv", index=False)
print("\nDescriptive comparison on test_site_full (do not use to reselect noise):")
print(comparison.to_string(index=False, float_format=lambda z: f"{z:.4f}"))
print("\nTest prevalence:", float(np.mean(ytest)), "| 3x prevalence utility reference:", 3*float(np.mean(ytest)))


## 5. Closure checks and report

The acceptance result is allowed to be negative: if every configuration with `epsilon <= 10` has weak predictive utility, report that no useful DP configuration was demonstrated under this cap rather than silently relaxing privacy after seeing test results. The test set has been used for prior Phase 3 baselines, so this is a validation-locked follow-up on the same held-out benchmark, not an untouched external replication.

In [ ]:
checks = {}
def record_check(name, passed, detail=""):
    checks[name] = {"status": "PASS" if bool(passed) else "FAIL", "detail": str(detail)}
    print(f"[{'PASS' if passed else 'FAIL'}] {name}: {detail}")

record_check("CLOSE-1 all three seeds swept on validation", SWEEP["seed"].nunique() == 3,
             f"seeds={sorted(SWEEP['seed'].unique().tolist())}; rows={len(SWEEP)}")
record_check("CLOSE-2 selected config meets epsilon cap", privacy_epsilon(selected_sigma)[0] <= EPSILON_CAP,
             f"sigma={selected_sigma}; epsilon={privacy_epsilon(selected_sigma)[0]:.4f}; cap={EPSILON_CAP}")
record_check("CLOSE-3 test rows absent from selection table", "used_test_data" not in SWEEP.columns,
             "sweep contains only train/validation-derived scores and validation-machine events")
record_check("CLOSE-4 final test uses held-out factory only", set(mtest["client_id"].unique()) == {HELD_OUT},
             f"clients={sorted(mtest['client_id'].unique().tolist())}")
record_check("CLOSE-5 final DP scores finite", np.isfinite(FINAL["test_site_full_ap"]).all(),
             f"seeds={FINAL['seed'].nunique()}")
record_check("CLOSE-6 checkpoints locked before test", (OUT_DIR / "reports" / "dp_selection_lock.json").exists(),
             "selection lock saved before loading test_site_full")
if any(v["status"] != "PASS" for v in checks.values()):
    raise AssertionError("Privacy closure checks failed; inspect the report before treating Phase 3 as closed.")

mean_ap = float(FINAL["test_site_full_ap"].mean())
prevalence = float(FINAL["test_prevalence"].iloc[0])
utility_floor = 3.0 * prevalence
fedavg_mean = float(comparison.loc[comparison["method"] == "FedAvg", "mean_ap"].iloc[0]) if (comparison["method"] == "FedAvg").any() else float("nan")
utility_status = "DP selected model outperforms mean FedAvg" if mean_ap > fedavg_mean else "DP selected model does not outperform mean FedAvg"
report = {
    "phase": "Phase 3 privacy closure",
    "input_artifacts": {"preparation": str(PREP_DIR), "phase2": str(P2_DIR), "phase3_v2": str(P3_DIR)},
    "output_directory": str(OUT_DIR),
    "selection_protocol": "Across seeds 42,43,44, choose noise multiplier with highest mean validation AP among candidates whose scoped epsilon bound <= 10; never use test metrics for noise selection.",
    "epsilon_cap": EPSILON_CAP, "delta": DELTA, "selected_noise_multiplier": selected_sigma,
    "epsilon_bound_training_update_channel": privacy_epsilon(selected_sigma)[0],
    "selected_mean_validation_ap": float(eligible.loc[np.isclose(eligible["noise_multiplier"], selected_sigma), "mean_validation_ap"].iloc[0]),
    "final_test_site_full_mean_ap": mean_ap, "final_test_site_full_sd_ap": float(FINAL["test_site_full_ap"].std(ddof=1)),
    "test_prevalence": prevalence, "contextual_utility_reference_3x_prevalence_not_used_for_selection": utility_floor,
    "utility_status": utility_status, "seeds": SEEDS, "rounds": MAX_ROUNDS,
    "checks": checks,
    "privacy_scope": "Client-level add/remove, clipped client update, Gaussian noise on aggregate; all four clients participate each round; no sampling amplification; trusted server; RDP upper bound only for training-update channel.",
    "limitations": [
        "The epsilon cap of 10 is a study-specific engineering criterion, not a universal privacy standard.",
        "The DP sweep is limited to full features/global normalization and three random seeds.",
        "Validation-derived checkpoint/threshold selection, normalization statistics, raw-data access and external diagnostic releases are outside the reported DP scope.",
        "The same test_site_full set was already used for prior Phase 3 baseline reporting; this is not an untouched external replication.",
        "A test PR-AUC below three times the prevalence reference is documented as insufficient utility for this project's closure target, not as a universal ML threshold.",
        "SecureAggSim is not cryptographic secure aggregation; the maintenance knowledge graph and grounded LLM remain later project phases."
    ],
}
(OUT_DIR / "reports" / "phase3_privacy_closure_report.json").write_text(json.dumps(report, indent=2, default=str))
md = [
    "# FedKG-Maint Phase 3 — Privacy Closure Report", "",
    f"- Selected noise multiplier: **{selected_sigma}**",
    f"- Scoped client-update epsilon bound: **{privacy_epsilon(selected_sigma)[0]:.3f}** at delta={DELTA}",
    f"- Mean validation AP: **{report['selected_mean_validation_ap']:.4f}**",
    f"- Mean test_site_full AP across seeds: **{mean_ap:.4f}** (SD={report['final_test_site_full_sd_ap']:.4f})",
    f"- Test prevalence: {prevalence:.4f}; three-times-prevalence reference (descriptive only): {utility_floor:.4f}",
    f"- Utility decision: **{utility_status}**", "",
    "## Validation-only noise sweep", "",
    "| Noise | Epsilon bound | Mean validation AP | SD | Eligible (epsilon cap) |", "|---:|---:|---:|---:|:---:|"
]
for _, r in SWEEP_SUMMARY.iterrows():
    md.append(f"| {r['noise_multiplier']:.2f} | {r['epsilon']:.3f} | {r['mean_validation_ap']:.4f} | {r['sd_validation_ap']:.4f} | {'yes' if r['eligible_epsilon_cap'] else 'no'} |")
md += ["", "## Final selected-configuration test results", "", FINAL.to_markdown(index=False, floatfmt=".4f"), "", "## Existing Phase 3 references", "", comparison.to_markdown(index=False, floatfmt=".4f"), "", "## Integrity checks", ""]
for k,v in checks.items(): md.append(f"- **{v['status']}** — {k}: {v['detail']}")
md += ["", "## Privacy scope and limitations", ""] + [f"- {x}" for x in report["limitations"]]
(OUT_DIR / "reports" / "phase3_privacy_closure_report.md").write_text("\n".join(md))
print("\n" + "="*72)
print("PHASE 3 PRIVACY CLOSURE CHECKS PASSED")
print("Descriptive conclusion:", utility_status)
print("Report:", OUT_DIR / "reports" / "phase3_privacy_closure_report.md")
print("Outputs:", OUT_DIR / "results")


## Interpretation and closure rules

- The privacy sweep chooses noise by **validation only** under the stated epsilon cap; no test metric changes the selected multiplier.
- If no DP configuration satisfies the utility criterion, close Phase 3 with a documented negative finding: the current client-update DP mechanism is implemented and accounted for within its scope, but useful predictive performance was not demonstrated under this cap and model/client setup.
- Do not claim end-to-end differential privacy. Preprocessing statistics, centralized validation-based selection, raw data access, and externally released diagnostics are not protected by this accountant.
- `SecureAggSim-FedAvg` demonstrates mask cancellation only. It is not a cryptographic protocol or a standalone security guarantee.
- This closes only Phase 3's federated-learning/privacy experiments. Knowledge Graph and grounded LLM explanations remain unimplemented project requirements.